# 40.22 Исследовательский прогон ТТРКГ и идентифицируемости

**Статус результата:** `exploratory_hypothesis_not_validated`.

Ноутбук выполняет доступные части серий `40–50` без обхода строгих контрактов:
строит кандидатные ансамбли ТТРКГ экспериментов 2 и 3, описывает тест
переключения каналов РНЦХ в исходных единицах, перебирает сценарные тканевые
чувствительности и проверяет ранг соответствующей модельной матрицы.

FEM-операторы отсутствуют. Поэтому коэффициенты переноса ниже являются осями
алгебраического стресс-теста, а не рассчитанными картами чувствительности.


## Границы расчёта

- Для обоих приборов временно используются `gain=1`, `sign=+1` и указанный в
  конфигурации перевод `mΩ → Ω`; относительный сигнал делится на сырой `BASE_1`.
  Связь шкал `RHEO_1` и `BASE_1` не калибрована.
- Эксперимент 2 использует первичные кандидатные R-зубцы. Для основной записи
  эксперимента 3 используется диагностический набор из 88 кандидатов, потому
  что первичный набор имеет известный длинный пропуск. Это решение не принимает
  ЭКГ-разметку.
- Тканевые кривые эксперимента 2 переносятся в эксперимент 3 для того же
  добровольца, но между несинхронными сессиями и разными приборами.
- Перебираются пары коэффициентов мягких тканей и лёгкого из множества
  `{-1, 0, +1}`. Отрицательные значения допустимы математически для
  тетраполярной чувствительности; сами выбранные числа не имеют статуса
  анатомического диапазона.
- В агрегат эксперимента 2 включаются только записи, где `BASE_1 > 5 Ом` не
  менее чем в 95 % выбранной задержки дыхания. Оставшиеся записи усредняются
  с равным весом; это отдельное исследовательское допущение.
- Сердечный коэффициент в сценарной матрице нормирован к единице только для
  проверки ранга. Это не FEM-производная по объёму сердца.


In [1]:
# Конфигурации, артефакт 33.06 и общие функции
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from exploratory_analysis import (
    EXPLORATORY_STATUS,
    aggregate_waveforms,
    interval_median,
    normalized_modes,
    pulse_ensemble,
    residual_scenario,
    select_rpeaks,
    source_design_diagnostics,
)
from two_layer_model import evaluate, geometry_from_size

EXP02_CONFIG_PATH = Path(os.environ["KALMYKOV_EXP02_CONFIG"]).expanduser().resolve()
EXP03_CONFIG_PATH = Path(os.environ["KALMYKOV_EXP03_CONFIG"]).expanduser().resolve()
EXP02 = json.loads(EXP02_CONFIG_PATH.read_text(encoding="utf-8"))
EXP03 = json.loads(EXP03_CONFIG_PATH.read_text(encoding="utf-8"))
DATA02 = Path(EXP02["data_root"]).expanduser().resolve()
DATA03 = Path(EXP03["data_root"]).expanduser().resolve()
DERIVED = Path(EXP02["derived_root"]).expanduser().resolve()
if DERIVED != Path(EXP03["derived_root"]).expanduser().resolve():
    raise RuntimeError("Для совместного прогона нужен один derived_root")

SIDE_ARTIFACT_PATH = DERIVED / "exp02" / "exploratory" / "33.06_side_arrays_exploratory.json"
SIDE = json.loads(SIDE_ARTIFACT_PATH.read_text(encoding="utf-8"))
if SIDE.get("status") != EXPLORATORY_STATUS:
    raise RuntimeError("33.06 должен быть отдельным исследовательским артефактом")

OUT_DIR = DERIVED / "cross_experiment" / "exploratory"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "40.22_ttrkg_transfer_identifiability_exploratory.json"
SENSITIVITY_VALUES = (-1.0, 0.0, 1.0)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def json_ready(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {key: json_ready(child) for key, child in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(child) for child in value]
    return value


def resolve(root, relative_path):
    path = (root / relative_path).resolve()
    path.relative_to(root)
    if not path.is_file():
        raise FileNotFoundError(relative_path)
    return path


def interpolate_rows(source_grid, values, target_grid):
    source_grid = np.asarray(source_grid, dtype=float)
    target_grid = np.asarray(target_grid, dtype=float)
    return np.asarray([np.interp(target_grid, source_grid, row) for row in np.asarray(values)])


def side_fractional_row(static_fit, state, size_m=0.14):
    rho1 = float(static_fit["rho1_ohm_m"])
    rho2 = float(static_fit[f"rho2_{state}_ohm_m"])
    a, b = geometry_from_size(float(size_m))
    result = evaluate(rho1, rho2, float(static_fit["h_m"]), a, b)
    return np.asarray([
        result.d_rho1 * rho1 / result.z,
        result.d_rho2 * rho2 / result.z,
        0.0,
    ])


In [2]:
# Кандидатные ансамбли ТТРКГ эксперимента 2
BREATH02 = DERIVED / "exp02" / "annotations" / "breathing"
ECG02 = DERIVED / "exp02" / "annotations" / "ecg"
MODE02 = {"задержка_вдох": "inhale", "задержка_выдох": "exhale"}
records02 = {}
provenance02 = []
for breathing_path in sorted(BREATH02.glob("*.json")):
    breathing = json.loads(breathing_path.read_text(encoding="utf-8"))
    if breathing.get("annotation_type") != "breathing":
        continue
    record_id = breathing["record_id"]
    ecg_path = ECG02 / f"{record_id}.json"
    ecg = json.loads(ecg_path.read_text(encoding="utf-8"))
    source_path = resolve(DATA02, breathing["input"]["relative_path"])
    source_sha = sha256_file(source_path)
    if source_sha != breathing["input"]["sha256"] or source_sha != ecg["input"]["sha256"]:
        raise RuntimeError(f"Конфликт SHA-256 для {record_id}")
    frame = pd.read_csv(source_path, encoding="utf-8")
    time_s = frame["TIME_s"].to_numpy(dtype=float)
    modes, selected_modes = normalized_modes(breathing)
    rpeaks = select_rpeaks(ecg, prefer_diagnostic=False)
    record = {
        "record_id": record_id,
        "size_mm": int(breathing["size_mm"]),
        "time_s": time_s,
        "rheo_1_mohm": frame["RHEO_1_mΩ"].to_numpy(dtype=float),
        "base_1_ohm": frame["BASE_1_Ω"].to_numpy(dtype=float),
        "modes": modes,
        "rpeaks_s": rpeaks.values,
    }
    records02.setdefault(breathing["subject_id"], []).append(record)
    provenance02.append({
        "record_id": record_id,
        "source_csv_sha256": source_sha,
        "breathing_sidecar_sha256": sha256_file(breathing_path),
        "breathing_qc_status": selected_modes.qc_status,
        "breathing_field": selected_modes.source_field,
        "ecg_sidecar_sha256": sha256_file(ecg_path),
        "ecg_qc_status": rpeaks.qc_status,
        "ecg_field": rpeaks.source_field,
    })

results02 = {}
for subject_id, records in sorted(records02.items()):
    records = sorted(records, key=lambda item: item["size_mm"])
    subject = {"modes": {}}
    side_subject = SIDE["subjects"][subject_id]
    static_fit = side_subject["static_h_profile"]["best"]
    for mode_name, state in MODE02.items():
        fractional_waveforms = []
        record_outputs = []
        grid = None
        for record in records:
            interval = record["modes"][mode_name]
            margin = float(EXP02["ttrkg_analysis"].get("hold_margin_s", 0.0))
            active_mask = (
                (record["time_s"] >= interval[0] + margin)
                & (record["time_s"] <= interval[1] - margin)
            )
            active_fraction = float(np.mean(
                record["base_1_ohm"][active_mask]
                > float(EXP02["record_qc"]["active_channel_threshold_ohm"])
            ))
            if active_fraction < 0.95:
                record_outputs.append({
                    "record_id": record["record_id"],
                    "size_mm": record["size_mm"],
                    "included_in_ttrkg_aggregate": False,
                    "channel_1_active_fraction_in_mode": active_fraction,
                    "exclusion_reason": "BASE_1_active_fraction_below_0.95",
                })
                continue
            ensemble = pulse_ensemble(
                record["time_s"], record["rheo_1_mohm"], record["rpeaks_s"],
                interval, EXP02["ttrkg_analysis"], sign=1.0, gain=1.0,
            )
            base = interval_median(
                record["time_s"], record["base_1_ohm"], interval
            )
            fractional = ensemble["mean"] / base
            fractional_waveforms.append(fractional)
            grid = ensemble["grid_s"] if grid is None else grid
            if not np.array_equal(grid, ensemble["grid_s"]):
                raise RuntimeError("Временные сетки эксперимента 2 не совпадают")
            record_outputs.append({
                "record_id": record["record_id"],
                "size_mm": record["size_mm"],
                "included_in_ttrkg_aggregate": True,
                "channel_1_active_fraction_in_mode": active_fraction,
                "base_1_scenario_ohm": base,
                "fractional_waveform": fractional,
                "n_beats": ensemble["n_beats"],
                "n_rejected": ensemble["n_rejected"],
            })
        if not fractional_waveforms:
            raise RuntimeError(f"Нет активных записей ТТРКГ для {subject_id}, {mode_name}")
        aggregate = aggregate_waveforms(fractional_waveforms)
        tissue = side_subject["dynamic"][mode_name]
        tissue_fractional = interpolate_rows(
            tissue["grid_s"],
            [tissue["fractional_rho1"], tissue["fractional_rho2"]],
            grid,
        )
        side_row = side_fractional_row(static_fit, state)
        scenarios = []
        for soft in SENSITIVITY_VALUES:
            for lung in SENSITIVITY_VALUES:
                residual = residual_scenario(aggregate["mean"], tissue_fractional, [soft, lung])
                rank_one = source_design_diagnostics([soft, lung, 1.0])
                rank_two = source_design_diagnostics([soft, lung, 1.0], side_row)
                scenarios.append({
                    "s_soft": soft,
                    "s_lung": lung,
                    **residual,
                    "one_ttrkg_channel_rank": rank_one["rank"],
                    "one_ttrkg_channel_nullity": rank_one["nullity"],
                    "ttrkg_plus_one_side_channel_rank": rank_two["rank"],
                    "ttrkg_plus_one_side_channel_nullity": rank_two["nullity"],
                })
        subject["modes"][mode_name] = {
            "state": state,
            "grid_s": grid,
            "records": record_outputs,
            "aggregate_fractional": aggregate,
            "tissue_fractional_from_33_06": tissue_fractional,
            "representative_side_fractional_row_140mm": side_row,
            "transfer_scenarios": scenarios,
        }
    results02[subject_id] = subject


In [3]:
# Основная дыхательная запись и тест переключения эксперимента 3
CANONICAL03 = [
    "time_s", "rheo_1_mohm", "base_1_ohm", "qs_1_ohm",
    "ecg_v", "rheo_2_mohm", "base_2_ohm", "qs_2_ohm",
]
spec_by_id = {item["record_id"]: item for item in EXP03["recordings"]}


def read_exp03(spec):
    path = resolve(DATA03, spec["relative_path"])
    frame = pd.read_csv(path)
    if list(frame.columns) != EXP03["source_columns"]:
        raise ValueError(f"Схема CSV не совпадает: {spec['record_id']}")
    frame.columns = CANONICAL03
    return path, frame.apply(pd.to_numeric, errors="raise")


main_spec = spec_by_id["exp03_both_breathing"]
main_path, main_frame = read_exp03(main_spec)
breathing03_path = DERIVED / "exp03" / "annotations" / "breathing" / "exp03_both_breathing.json"
ecg03_path = DERIVED / "exp03" / "annotations" / "ecg" / "exp03_both_breathing.json"
breathing03 = json.loads(breathing03_path.read_text(encoding="utf-8"))
ecg03 = json.loads(ecg03_path.read_text(encoding="utf-8"))
if sha256_file(main_path) != breathing03["input"]["sha256"] or sha256_file(main_path) != ecg03["input"]["sha256"]:
    raise RuntimeError("Конфликт SHA-256 основной записи эксперимента 3")
modes03, selected_modes03 = normalized_modes(breathing03)
rpeaks03 = select_rpeaks(ecg03, prefer_diagnostic=True)

MODE03 = {
    "вдох_и_задержка": ("задержка_вдох", "inhale"),
    "задержка_на_выдохе": ("задержка_выдох", "exhale"),
}
result03 = {"record_id": "exp03_both_breathing", "modes": {}}
side_nik = SIDE["subjects"]["exp02_nik"]
for mode03, (mode02, state) in MODE03.items():
    ensemble = pulse_ensemble(
        main_frame["time_s"].to_numpy(dtype=float),
        main_frame["rheo_1_mohm"].to_numpy(dtype=float),
        rpeaks03.values,
        modes03[mode03],
        EXP03["ttrkg_analysis"],
        sign=1.0,
        gain=1.0,
    )
    base = interval_median(
        main_frame["time_s"].to_numpy(dtype=float),
        main_frame["base_1_ohm"].to_numpy(dtype=float),
        modes03[mode03],
    )
    measured = ensemble["mean"] / base
    tissue = side_nik["dynamic"][mode02]
    tissue_fractional = interpolate_rows(
        tissue["grid_s"],
        [tissue["fractional_rho1"], tissue["fractional_rho2"]],
        ensemble["grid_s"],
    )
    static_fit_nik = side_nik["static_h_profile"]["best"]
    side_row = side_fractional_row(static_fit_nik, state)
    scenarios = []
    for soft in SENSITIVITY_VALUES:
        for lung in SENSITIVITY_VALUES:
            residual = residual_scenario(measured, tissue_fractional, [soft, lung])
            rank_one = source_design_diagnostics([soft, lung, 1.0])
            rank_two = source_design_diagnostics([soft, lung, 1.0], side_row)
            scenarios.append({
                "s_soft": soft,
                "s_lung": lung,
                **residual,
                "one_ttrkg_channel_rank": rank_one["rank"],
                "one_ttrkg_channel_nullity": rank_one["nullity"],
                "ttrkg_plus_one_side_channel_rank": rank_two["rank"],
                "ttrkg_plus_one_side_channel_nullity": rank_two["nullity"],
            })
    result03["modes"][mode03] = {
        "state": state,
        "source_tissue_mode_exp02": mode02,
        "grid_s": ensemble["grid_s"],
        "n_beats": ensemble["n_beats"],
        "n_rejected": ensemble["n_rejected"],
        "base_1_scenario_ohm": base,
        "measured_fractional": measured,
        "tissue_fractional_transferred_from_exp02_nik": tissue_fractional,
        "transfer_scenarios": scenarios,
    }

switch_spec = spec_by_id["exp03_switch_test"]
switch_path, switch_frame = read_exp03(switch_spec)
threshold = float(EXP03["active_channel_threshold_ohm"])
active_1 = switch_frame["base_1_ohm"].to_numpy(dtype=float) > threshold
active_2 = switch_frame["base_2_ohm"].to_numpy(dtype=float) > threshold
labels = np.where(active_1 & active_2, "both", np.where(active_1, "channel_1_only", np.where(active_2, "channel_2_only", "none")))
dt_s = float(np.median(np.diff(switch_frame["time_s"].to_numpy(dtype=float))))
switch_states = {}
for label in sorted(set(labels.tolist())):
    mask = labels == label
    switch_states[label] = {
        "duration_s": float(mask.sum() * dt_s),
        "sample_count": int(mask.sum()),
        "base_1_median_raw_ohm": float(np.median(switch_frame.loc[mask, "base_1_ohm"])),
        "base_2_median_raw_ohm": float(np.median(switch_frame.loc[mask, "base_2_ohm"])),
        "rheo_1_median_raw_mohm": float(np.median(switch_frame.loc[mask, "rheo_1_mohm"])),
        "rheo_2_median_raw_mohm": float(np.median(switch_frame.loc[mask, "rheo_2_mohm"])),
    }

def extreme_repeat_fraction(values):
    values = np.asarray(values, dtype=float)
    return float(((values == values.min()) | (values == values.max())).mean())

hardware_qc03 = {
    "main_rheo_1_extreme_repeat_fraction": extreme_repeat_fraction(main_frame["rheo_1_mohm"]),
    "main_rheo_2_extreme_repeat_fraction": extreme_repeat_fraction(main_frame["rheo_2_mohm"]),
    "switch_rheo_1_extreme_repeat_fraction": extreme_repeat_fraction(switch_frame["rheo_1_mohm"]),
    "switch_rheo_2_extreme_repeat_fraction": extreme_repeat_fraction(switch_frame["rheo_2_mohm"]),
    "interpretation": "descriptive_exact_extreme_repetition_not_proof_of_saturation",
}


In [4]:
# Сохранение исследовательского артефакта 40–50
artifact = {
    "schema_version": 1,
    "status": EXPLORATORY_STATUS,
    "artifact_id": "40.22_ttrkg_transfer_identifiability_exploratory",
    "strict_pipeline_authorized": False,
    "fem_operator_used": False,
    "configuration_sha256": {
        "exp02": sha256_file(EXP02_CONFIG_PATH),
        "exp03": sha256_file(EXP03_CONFIG_PATH),
    },
    "side_artifact_sha256": sha256_file(SIDE_ARTIFACT_PATH),
    "scenario": {
        "rheo_unit_scale_to_ohm": 0.001,
        "rheo_gain": 1.0,
        "rheo_sign": 1,
        "base_gain": 1.0,
        "base_sign": 1,
        "base_offset_ohm": 0.0,
        "tissue_sensitivity_values": list(SENSITIVITY_VALUES),
        "heart_column_normalization": 1.0,
        "interpretation": "algebraic_stress_test_not_fem_sensitivity",
    },
    "exp02": {
        "input_provenance": provenance02,
        "subjects": results02,
    },
    "exp03": {
        "input_provenance": {
            "main_csv_sha256": sha256_file(main_path),
            "breathing_sidecar_sha256": sha256_file(breathing03_path),
            "breathing_qc_status": selected_modes03.qc_status,
            "breathing_field": selected_modes03.source_field,
            "ecg_sidecar_sha256": sha256_file(ecg03_path),
            "ecg_qc_status": rpeaks03.qc_status,
            "ecg_field": rpeaks03.source_field,
            "switch_csv_sha256": sha256_file(switch_path),
        },
        "main_record": result03,
        "channel_switch_raw_state_summary": switch_states,
        "hardware_qc": hardware_qc03,
    },
    "limitations": [
        "candidate_annotations_are_not_accepted",
        "instrument_gain_sign_and_relative_base_pulse_scale_are_not_calibrated",
        "no_fem_ttrkg_sensitivity_operator",
        "exp02_to_exp03_tissue_transfer_is_cross_session_and_not_accepted",
        "one_ttrkg_channel_has_rank_one_for_three_source_amplitudes",
        "one_ttrkg_plus_one_side_channel_has_at_most_rank_two_for_three_sources",
        "rank_of_a_scenario_matrix_does_not_validate_its_physics",
        "exp02_ttrkg_records_with_channel_1_active_fraction_below_0_95_are_excluded",
        "remaining_exp02_ttrkg_records_are_aggregated_with_equal_weight",
        "no_full_uncertainty_or_temporal_covariance_model",
    ],
}
OUT_PATH.write_text(json.dumps(json_ready(artifact), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

summary = {
    "exp02_subjects": sorted(results02),
    "exp02_records": len(provenance02),
    "exp02_included_records_by_subject_mode": {
        subject_id: {
            mode_name: sum(
                item.get("included_in_ttrkg_aggregate", False)
                for item in mode["records"]
            )
            for mode_name, mode in subject["modes"].items()
        }
        for subject_id, subject in results02.items()
    },
    "exp03_ecg_field": rpeaks03.source_field,
    "exp03_beats": {name: item["n_beats"] for name, item in result03["modes"].items()},
    "switch_states": {name: item["duration_s"] for name, item in switch_states.items()},
    "rank_conclusion": "1 TTRKG channel: rank 1/3; + one side channel: rank at most 2/3",
}
included_rows = []
for subject_id, subject in results02.items():
    for mode_name, mode in subject["modes"].items():
        included_rows.append({
            "subject_id": subject_id,
            "mode": mode_name,
            "included_records": sum(
                item.get("included_in_ttrkg_aggregate", False)
                for item in mode["records"]
            ),
            "total_records": len(mode["records"]),
        })
switch_table = (
    pd.DataFrame.from_dict(switch_states, orient="index")
    .rename_axis("state")
    .reset_index()
)
rank_table = pd.DataFrame([
    {"измерения": "1 ТТРКГ-канал", "максимальный_ранг": 1, "неизвестных_источников": 3},
    {"измерения": "ТТРКГ + 1 боковая сборка", "максимальный_ранг": 2, "неизвестных_источников": 3},
])
print("Исследовательский артефакт:", OUT_PATH.name)
print("Эксперимент 2: записи, вошедшие в ансамбли ТТРКГ")
display(pd.DataFrame(included_rows))
print("Эксперимент 3: число сердечных циклов по режимам")
display(pd.DataFrame([
    {"mode": name, "n_beats": item["n_beats"]}
    for name, item in result03["modes"].items()
]))
print("Тест переключения каналов РНЦХ в исходных единицах")
display(switch_table.round(4))
print("Структурная идентифицируемость сценарной модели")
display(rank_table)


Исследовательский артефакт: 40.22_ttrkg_transfer_identifiability_exploratory.json
Эксперимент 2: записи, вошедшие в ансамбли ТТРКГ


,subject_id,mode,included_records,total_records
0,exp02_georg,задержка_вдох,8,10
1,exp02_georg,задержка_выдох,9,10
2,exp02_nik,задержка_вдох,9,9
3,exp02_nik,задержка_выдох,9,9


Эксперимент 3: число сердечных циклов по режимам


,mode,n_beats
0,вдох_и_задержка,17
1,задержка_на_выдохе,20


Тест переключения каналов РНЦХ в исходных единицах


,state,duration_s,sample_count,base_1_median_raw_ohm,base_2_median_raw_ohm,rheo_1_median_raw_mohm,rheo_2_median_raw_mohm
0,both,26.330,5266,54.189,40.840,-66.4275,46.0010
1,channel_1_only,8.780,1756,92.693,0.000,27.2695,-145.8455
2,channel_2_only,10.915,2183,0.000,36.883,-31.0810,-5.5870


Структурная идентифицируемость сценарной модели


,измерения,максимальный_ранг,неизвестных_источников
0,1 ТТРКГ-канал,1,3
1,ТТРКГ + 1 боковая сборка,2,3


## Интерпретация

Различие остаточных кривых между девятью парами `S_soft`, `S_lung` показывает,
насколько вывод зависит от отсутствующего FEM-оператора. Ни одна пара не
выбирается как лучшая по данным.

Один ТТРКГ-канал даёт одну строку для трёх условных источников и имеет ранг 1.
Добавление одной боковой сборки даёт не более двух независимых строк. Поэтому
без дополнительных физических ограничений или заранее проверенных тканевых
операторов три произвольных источника не разделяются. Это структурный вывод о
сценарной матрице, а не доказательство состава реального сигнала.

Сводка теста переключения фиксирует только изменение записанных каналов при
разных состояниях подключения. Причина эффекта не устанавливается.
